# =========================================================
# Component: Memory
# State management of tools / agent
# ========================================================

In [ ]:
# **************************
# (example 1) simple history
# **************************

import os
from langchain_openai import ChatOpenAI
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.messages import AIMessage, HumanMessage

llm = ChatOpenAI() # add the api_key parameter here

history = ChatMessageHistory()

In [ ]:
history.add_user_message("hello") # user's first message
history.add_ai_message("Greetings. How are you ?") # AI's reply

history.add_user_message("i am fine. today i want to talk about some interesting topics") # user message
history.add_ai_message("Nice. Looking forward to this.") # AI reply

print(history)

In [ ]:
history.messages

In [ ]:
# *******************************************
# (example 2) : Simulation of a real-life chat
# ********************************************
history = ChatMessageHistory()

# Start the chat loop
msg = ''' Hi. I am your Tax Consultant. How can i help you?
        To end this conversation, type exit/bye/quit/close
        '''
flag = True
exit_val = ["exit","quit","bye","close"]

while flag:
    user_input = input("You: ")
    if any(ev for ev in exit_val if ev in user_input):
        flag = False

    # Add user message to history
    history.add_user_message(user_input)

    # Generate AI response using chat history
    ai_response = llm.invoke(history.messages)

    # Display and store AI response
    print(f"User: {user_input}")
    print(f"AI: {ai_response.content}")
    history.add_ai_message(ai_response.content)

In [ ]:
# total messages
print(f"There are {len(history.messages)} messages")

In [ ]:
# display the full chat history
print("\n--- Chat History ---")
for msg in history.messages:
    role = "User" if isinstance(msg, HumanMessage) else "AI"
    print(f"{role}: {msg.content}")
    print("*" * 50)

In [ ]:
# Answer for a particular Question number
    
q = 2

print(f"Question Number: {q}")
print(f"Q:" + history.messages[q].content)
print()
print(f"A:" + history.messages[q+1].content)


# A simple langchain example to simulate a Manual entity memory.
# Attached dataset (CSV) is the data source.
# All writes in MySQL table

Entity Memory in LangChain is a specialized memory module that extracts, stores, and updates facts about specific entities (people, places, organizations, objects) mentioned during a conversation.

In [ ]:
import os
import pandas as pd
import mysql.connector
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
csv_path = r"D:\stackroute\2_AI-assisted-programming\learning_requirements\ey\2026\dataset\household_energy_requirement.csv"

In [ ]:
energy_data = pd.read_csv(csv_path)
print(energy_data.head())

In [ ]:
# Connect to the MySQL database
conn = mysql.connector.connect(host="localhost", user="root", password="root", database="ey")
print(conn)

In [ ]:
energy_data.columns

In [ ]:
# The entity data is the memory data that comes from the CSV

entities = [ "entid", "house_type", "num_occupants", "has_ac", "num_ac_units", "daily_energy_consumption_kWh", 
            "monthly_energy_consumption_kWh","estimated_peak_load_kW"]

entid = "ENT100"
ent_data = energy_data[entities][energy_data.entid == entid]

# Convert the output into a JSON format
ent_data = ent_data.iloc[0].to_dict()
print(ent_data)

In [ ]:
random_rec = energy_data.sample(1)
print(random_rec.entid.tolist()[0])

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = PromptTemplate(input_variables=["ent_data", "question"],
    template="""
You are an energy advisor.

Use the following stored entity memory:

{ent_data}

Question:
{question}

Answer in simple terms.
"""
)

chain = prompt | llm | StrOutputParser()

In [ ]:
query = "Is this household likely to have high energy consumption? Explain why"

response = chain.invoke({ "ent_data": ent_data, "question": query})
print(response)

In [ ]:
# Insert the response in the MySQL

def ExecuteQuery(action, query, values=None):
    ret = {"status": '', "message": "", "record": ""}
    act_msg = ''
    CURSOR = conn.cursor(dictionary=True)

    try:
        if action == "I":
            act_msg = "Inserted"
        elif action == "U":
            act_msg = "Updated"
        elif action == "D":
            act_msg = "Deleted"
        elif action == "S":
            act_msg = "Retrieved"

        if action in ["I", "U", "D"]:
            CURSOR.execute(query, values)
            conn.commit()
            ret["status"] = "SUCCESS"
            ret["message"] = f"{CURSOR.rowcount} Record(s) {act_msg}"
            ret["record"] = ''
        elif action == "S":
            if values is not None:
                CURSOR.execute(query, values)
            else:
                CURSOR.execute(query)

            data = CURSOR.fetchall()
            ret["status"] = "SUCCESS"
            ret["message"] = f"{len(data)} Record(s) {act_msg} "
            ret["record"] = pd.DataFrame(data)

    except Exception as e:
        ret["status"] = "EXCEPTION"
        ret["message"] = str(e)
        ret["record"] = ''
        CURSOR.close()

    finally:
        CURSOR.close()

    return (ret)

In [ ]:
ent_data

In [ ]:
entity_values = ''

for d in ent_data.values():
    entity_values += str(d) + "~"

print(entity_values)

query = '''INSERT INTO entity_memory(entity_id, entity_name,memory_value) VALUES(%s,%s,%s) '''
values = (entid,entity_values,response)

In [ ]:
status = ExecuteQuery(action="I",query=query,values=values)

In [ ]:
print(status)

# LangChain Entity Memory simulation Example

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.memory import ConversationEntityMemory
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
# initialize memory
memory = ConversationEntityMemory(llm=llm)

In [ ]:
# define prompt template

prompt = PromptTemplate(
    input_variables=["input", "entities"],
    template="""
You are an energy advisor.

Here is what you know about entities so far:
{entities}

User query:
{input}

Answer in simple terms.
"""
)

In [ ]:
# build the Chain
chain = LLMChain(llm=llm, prompt=prompt, memory=memory)

In [ ]:
initial_fact = (
    "ENT100 is a household with 4 occupants, has AC, "
    "2 AC units, daily energy consumption of 18 kWh, "
    "monthly consumption of 540 kWh, and an estimated peak load of 4.2 kW."
)

chain.run(initial_fact)

In [ ]:
query = "Is this household likely to have high energy consumption? Explain why"
response = chain.run(query)
print(response)

# =========================================================
# Component: Vector Store
# Create embeddings on input data and perform Q/A
# ========================================================


In [ ]:
#  pip install faiss-cpu tiktoken

import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

In [ ]:
f1 = r"D:\stackroute\2_AI-assisted-programming\learning_requirements\ey\2026\dataset\ai.txt"

In [ ]:
# Step 1: Load text from file
loader = TextLoader(f1)
documents = loader.load()

# Step 2: Split into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=20)
docs = text_splitter.split_documents(documents)
print(f"There are {len(docs)} chunks")

In [ ]:
# Step 3: print the chunks and its length

chunks = [doc.page_content for doc in docs]

for ndx, c in enumerate(chunks,start=1):
    print(f"Chunk = {ndx}, Text = {c}")

In [ ]:
# Step 4: Convert to vector store using OpenAI
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_texts(chunks,embedding=embeddings)

In [ ]:
# Step 5: Perform similarity search
query = "Give 3 examples of real world AI"
results = vectorstore.similarity_search(query, k=2)

In [ ]:
# Step 6: Display results
for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---\n{doc.page_content}")

In [ ]:
embeddings_small = OpenAIEmbeddings(model="text-embedding-3-small")

word = "hello"
print(len(embeddings_small.embed_query(word)))

In [ ]:
# view the embededding

print(embeddings_small.embed_query(word))

In [ ]:
# Example 2

query = "how was AI founded"
results = vectorstore.similarity_search(query, k=1)

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---\n{doc.page_content}")

# =========================================================
# Component: Document Loader
# Read a document
# ========================================================

In [ ]:
# pip install PyPDF pypdf

from langchain_community.document_loaders import PyPDFLoader

In [ ]:
f1 = r"D:\stackroute\2_AI-assisted-programming\learning_requirements\ey\2026\dataset\indian_classical_music.pdf"

In [ ]:
loader = PyPDFLoader(f1)
data = loader.load()

print(data)
print(f"Total pages = {len(data)}")

In [ ]:
# print the individual pages
for i in range(len(data)):
    print(f"Page : {i+1}, Content: \n{data[i].page_content}")

In [ ]:
f1 = r"D:\stackroute\2_AI-assisted-programming\learning_requirements\ey\2026\dataset\photography.docx"

In [ ]:
# -----------------------------------
# read a Word document for processing
# -----------------------------------

# %pip install unstructured
# % pip install python-docx

from langchain_community.document_loaders import UnstructuredWordDocumentLoader

loader = UnstructuredWordDocumentLoader(f1)
documents = loader.load()

data = ''

for doc in documents:
    data+= doc.page_content

print(data)

In [39]:
# JSON file loader

# python -m pip install jq

from langchain_community.document_loaders import JSONLoader

In [ ]:
f = r"D:\stackroute\2_AI-assisted-programming\learning_requirements\ey\2026\dataset\employees.json"

loader = JSONLoader(file_path=f, jq_schema=".[]", text_content=False)

docs = loader.load()

for doc in docs:
    print(doc.page_content)

In [ ]:
# KAG